# 02 — Explore the Lung data

What is in the file before touching it: which ages, how many mice, how the cell types are spread across them. **The number of mice matters more than the number of cells**, because mice are the independent samples.

### 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

### 2. Load

In [ ]:
adata = sc.read_h5ad("../data/tabula-muris-senis-droplet-processed-official-annotations-Lung.h5ad")
if not Path("../data/lung.h5ad").exists():   # rewriting once silences the old-format warnings
    adata.write_h5ad("../data/lung.h5ad")

### 3. Overview: ages, mice, cell types

In [ ]:
print(adata)
print(adata.obs["age"].value_counts())
print(adata.obs.groupby("age", observed=True)["mouse.id"].nunique())   # mice per age
print(adata.obs["cell_ontology_class"].value_counts())

### 4. Is `.X` raw counts or processed?

Whole numbers mean raw counts. Normalizing data that is already normalized would be a silent error.

In [ ]:
print(adata.X[:5, :5].toarray(), "max:", adata.X.max())
print("has .raw:", adata.raw is not None)
if adata.raw is not None:
    print(adata.raw.X[:5, :5].toarray(), "raw max:", adata.raw.X.max())

### 5. Are mitochondrial genes present?

In [ ]:
print(adata.var_names[adata.var_names.str.lower().str.startswith("mt-")].tolist())

### 6. Cells per cell type, by age and by mouse

In [ ]:
print(pd.crosstab(adata.obs["cell_ontology_class"], adata.obs["age"]))
print(pd.crosstab(adata.obs["cell_ontology_class"], adata.obs["mouse.id"]))

### 7. Sex balance, counted in mice (not cells)

In [ ]:
mice = adata.obs[["mouse.id", "age", "sex"]].drop_duplicates()
print(pd.crosstab(mice["age"], mice["sex"]))

### 8. The provided UMAP

On the `mouse.id` panel, cells of each type should mix across mice. A cluster made mostly of one mouse is a warning sign.

In [ ]:
sc.pl.umap(adata, color=["cell_ontology_class", "age", "mouse.id"], ncols=1, show=False)
plt.savefig("../figure/lung_umap_provided.png", dpi=200, bbox_inches="tight")

### Findings

- **Size:** 24,540 cells, 20,138 genes, **16 mice**. Ages 1m, 3m, 18m, 21m, 30m (no 24m). Young (1m + 3m) = 6 mice, old = 10 mice.
- **`.X` is log-normalized** (max 10, likely clipped). **`.raw` holds raw counts** (integers, max 5,659).
- **No mitochondrial genes** in this release, so `pct_counts_mt` can't be used.
- **Some age differences come from single mice.** 30m is 45% of cells, mostly classical monocytes from 30-M-2. Alveolar macrophages at 1m are mostly from 1-M-63. Fibroblasts and adventitial cells come almost only from the 3m male samples.
- **Well mixed across mice (most trustworthy):** B cells, CD8 T cells, CD4 T cells, NK cells, bronchial smooth muscle.
- **Sex is partly confounded with age:** 1m and 30m are all male, 21m all female, 3m and 18m are 2 F + 2 M.
- **Pooled samples:** IDs like `3-M-5/6` probably mean two mice pooled into one sample. Each ID is treated as one unit.